# Free LLM API Test: Google AI Studio & NVIDIA build & OpenRouter

This course uses **cloud LLM APIs that are free to use**, with no local GPU or server. The default LLM is
**NVIDIA build** (`deepseek-ai/deepseek-v4.1-flash`) and the comparison target is **OpenRouter** (`openrouter/free`);
here we connect to and test all three, including (optionally) Google AI Studio.

| Service | Free | Get a key | Access (common factory) | Example model |
|---|---|---|---|---|
| **Google AI Studio** (optional) | ✅ Free tier | [aistudio.google.com](https://aistudio.google.com) | `utils.get_llm('google')` | `gemini-3.1-flash-lite` |
| **NVIDIA build** ⭐Default | ✅ Free credits | [build.nvidia.com](https://build.nvidia.com) | `utils.get_llm('nvidia')` | `deepseek-ai/deepseek-v4.1-flash`, etc. |
| **OpenRouter** ⭐Comparison | ✅ Free model router | [openrouter.ai](https://openrouter.ai) | `utils.get_llm('openrouter')` | `openrouter/free` (automatic free selection) |

> For all three services, `utils.get_llm()` returns the appropriate LangChain connector (`ChatGoogleGenerativeAI` / `ChatNVIDIA` /
> `ChatOpenAI`), so they can be used with the **same interface** in later notebooks.
> Because OpenRouter is OpenAI-compatible, we show both **calling the OpenRouter SDK (= OpenAI SDK) directly** and **connecting via LangChain**.

> 📦 Key setup and issuance steps: [`env_guides/M02_0_free_llm_api_eng.md`](env_guides/M02_0_free_llm_api_eng.md)

### Prerequisites
- Fill in `NVIDIA_API_KEY` and `OPENROUTER_API_KEY` in `notebooks/.env` (required). `GOOGLE_API_KEY` is optional.
- For services without a key, only **guidance on getting one** is printed, with no error.

> ⚠️ **Usage limits for OpenRouter free models**: New users receive a small free allowance for testing.
> Free models have low rate limits (**50 requests per day in total**), making them unsuitable for production.
> Topping up **$10 or more** in credits raises the free-model limit to **1,000 requests per day**.
> With `openrouter/free` (the **Free Models Router**), a free model is selected automatically for each request.

---
## 0. Environment Setup

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(''))  # add notebooks/ to the import path

import utils
utils.reload_env()  # reload .env (refresh keys, etc.)
from agentic_lib.bootstrap import to_text  # provider-agnostic response normalization (handles list content/<think>)

# Install the connectors for the three services (passes quickly if already installed)
#   - langchain-google-genai        : Google Gemini
#   - langchain-nvidia-ai-endpoints : NVIDIA build
#   - openai / langchain-openai     : OpenRouter (OpenAI-compatible) — direct SDK call + LangChain connection
utils.uv_install(['langchain-google-genai', 'langchain-nvidia-ai-endpoints',
                  'openai', 'langchain-openai'])

# Check key configuration status (loaded from .env)
GOOGLE_API_KEY     = os.getenv('GOOGLE_API_KEY', '')
NVIDIA_API_KEY     = os.getenv('NVIDIA_API_KEY', '')
OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY', '')

print('Google AI Studio key:', 'set' if GOOGLE_API_KEY else 'not set — aistudio.google.com')
print('NVIDIA build key   :', 'set' if NVIDIA_API_KEY else 'not set — build.nvidia.com')
print('OpenRouter key     :', 'set' if OPENROUTER_API_KEY else 'not set — openrouter.ai')

# Common test prompt
from langchain_core.messages import HumanMessage, SystemMessage
PROMPT = [SystemMessage(content='Answer concisely in English.'),
          HumanMessage(content='Explain Agentic AI in one sentence.')]

LLM 공급자: nvidia
  NVIDIA build Key: 설정됨  /  Model: meta/llama-3.1-8b-instruct


[uv] 설치 완료: ['langchain-google-genai', 'langchain-nvidia-ai-endpoints', 'openai', 'langchain-openai']
Google AI Studio 키: 설정됨
NVIDIA build 키   : 설정됨
OpenRouter 키     : 설정됨


---
## 1. Google AI Studio (Gemini) — Free Tier

1. Go to [aistudio.google.com](https://aistudio.google.com) → `Get API Key` → `Create API key`
2. Save `GOOGLE_API_KEY=AIza...` in `notebooks/.env`
3. `utils.get_llm('google')` returns `ChatGoogleGenerativeAI(model='gemini-3.1-flash-lite')`.

> Gemini sometimes returns `content` as a list, so we normalize it with `to_text()` before printing.

In [2]:
gemini_answer = None
if not GOOGLE_API_KEY:
    print('GOOGLE_API_KEY not set — get one at https://aistudio.google.com and add it to .env.')
else:
    try:
        gemini = utils.get_llm('google')          # ChatGoogleGenerativeAI(gemini-3.1-flash-lite)
        resp = gemini.invoke(PROMPT)
        gemini_answer = to_text(resp.content)      # normalize list content
        print('[Google Gemini response]')
        print(' ', gemini_answer)
    except Exception as e:
        print(f'[Google call failed] {type(e).__name__}: {e}')

[Google Gemini 응답]
  에이전틱 AI는 단순히 명령을 수행하는 것을 넘어, 스스로 목표를 설정하고 계획을 세워 자율적으로 문제를 해결하는 지능형 시스템입니다.


---
## 2. NVIDIA build (build.nvidia.com) — Free Credits

1. Sign in at [build.nvidia.com](https://build.nvidia.com) → the model page you want → `Get API Key` (free)
2. Save `NVIDIA_API_KEY=nvapi-...` in `notebooks/.env` (the model is `NVIDIA_MODEL`, default `deepseek-ai/deepseek-v4.1-flash`)
3. **`utils.get_llm('nvidia')`** returns `ChatNVIDIA` (`langchain-nvidia-ai-endpoints`).
   It uses the same `.invoke()` / `.stream()` interface as other LangChain models (the same way as OpenRouter/Google).

> Check available model names on each model page at build.nvidia.com (e.g. `deepseek-ai/deepseek-v4.1-flash`,
> `nvidia/nemotron-3-super-120b-a12b`, `z-ai/glm-5.3-flash`). Large models may respond slowly or be overloaded (503).
> Models may be retired without notice (410 Gone), so if you get an error, check the model page to see whether it is still offered.

In [3]:
# The model is set via NVIDIA_MODEL in utils/.env (default deepseek-ai/deepseek-v4.1-flash)
NVIDIA_MODEL = utils.NVIDIA_MODEL

nvidia_answer = None
if not NVIDIA_API_KEY:
    print('NVIDIA_API_KEY not set — get one at https://build.nvidia.com and add it to .env.')
else:
    try:
        nvidia = utils.get_llm('nvidia')          # returns ChatNVIDIA(model=NVIDIA_MODEL)
        resp = nvidia.invoke(PROMPT)
        nvidia_answer = to_text(resp.content)
        print(f'[NVIDIA build response · {NVIDIA_MODEL}]')
        print(' ', nvidia_answer)
    except Exception as e:
        print(f'[NVIDIA call failed] {type(e).__name__}: {e}')
        print('  → Check that the model name is offered on build.nvidia.com and that your key/credits are valid.')

[NVIDIA build 응답 · meta/llama-3.1-8b-instruct]
  에이전틱(Agentic) AI는 사용자의 의도와 목표를 달성하기 위해 행동하는 인공지능을 말합니다.


---
## 3. OpenRouter — Free Model Router (`openrouter/free`)

[OpenRouter](https://openrouter.ai) relays models from many providers through **a single OpenAI-compatible endpoint**
(`https://openrouter.ai/api/v1`). New users receive a small free allowance and can use the **free models**
marked with `:free`.

- **`openrouter/free` (Free Models Router)** — **automatically selects** an available free model for each request.
  You can also specify an individual free model directly (e.g. `deepseek/deepseek-chat-v3-0324:free`).
- **Free limits**: Free models have low rate limits — **50 requests per day in total**. Topping up **$10 or more** in credits gives **1,000 requests per day**.
- The key is `OPENROUTER_API_KEY` (`sk-or-...`) in `.env`.

Since OpenRouter is OpenAI-compatible, we show **both of the methods** below.

| Method | Code | Notes |
|---|---|---|
| **A. OpenRouter SDK** | `OpenAI(base_url='https://openrouter.ai/api/v1', api_key=...)` | Uses the OpenAI Python SDK as-is |
| **B. LangChain** | `utils.get_llm('openrouter')` (= `ChatOpenAI`) | Same `.invoke()`/`.stream()` interface as other notebooks |

> Reference: [Free Models Router Playground](https://openrouter.ai/docs/cookbook/get-started/free-models-router-playground)

In [4]:
# Method A) Call the OpenRouter SDK directly — OpenRouter is OpenAI-compatible, so we use the openai Python SDK as-is.
openrouter_answer = None
if not OPENROUTER_API_KEY:
    print('OPENROUTER_API_KEY not set — get one at https://openrouter.ai and add it to .env.')
else:
    try:
        from openai import OpenAI
        client = OpenAI(
            base_url='https://openrouter.ai/api/v1',    # OpenRouter endpoint (OpenAI-compatible)
            api_key=OPENROUTER_API_KEY,
        )
        completion = client.chat.completions.create(
            model='openrouter/free',                     # router that auto-selects a free model (Free Models Router)
            messages=[
                {'role': 'system', 'content': 'Answer concisely in English.'},
                {'role': 'user',   'content': 'Explain Agentic AI in one sentence.'},
            ],
            # (Optional) set only if you want your app listed in openrouter.ai rankings/usage stats
            extra_headers={'HTTP-Referer': 'https://localhost', 'X-Title': 'Agentic AI Tutorial'},
        )
        openrouter_answer = completion.choices[0].message.content
        # The free model the router actually chose can be seen in the response's model field
        print(f'[OpenRouter SDK response · model actually selected: {completion.model}]')
        print(' ', openrouter_answer)
    except Exception as e:
        print(f'[OpenRouter SDK call failed] {type(e).__name__}: {e}')
        print('  → Check your key/free limits (free models: 50 requests/day, 1000 requests with 10+ credits).')

[OpenRouter SDK 호출 실패] RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1785024000000'}, 'provider_name': None, 'previous_errors': [{'code': 429, 'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day'}, {'code': 429, 'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day'}, {'code': 429, 'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day'}, {'code': 429, 'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day'}, {'code': 429, 'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day

In [5]:
# Method B) LangChain connection — utils.get_llm('openrouter') returns ChatOpenAI(base_url=OpenRouter).
#          It can then be used with the same .invoke() / .stream() interface as in later notebooks.
openrouter_lc_answer = None
if not OPENROUTER_API_KEY:
    print('OPENROUTER_API_KEY not set — skipping the LangChain example.')
else:
    try:
        openrouter_llm = utils.get_llm('openrouter')   # ChatOpenAI(model='openrouter/free', base_url=...)
        resp = openrouter_llm.invoke(PROMPT)
        openrouter_lc_answer = to_text(resp.content)   # normalize the response
        print(f'[OpenRouter LangChain response · {utils.OPENROUTER_MODEL}]')
        print(' ', openrouter_lc_answer)
    except Exception as e:
        print(f'[OpenRouter LangChain call failed] {type(e).__name__}: {e}')

[OpenRouter LangChain 호출 실패] RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1785024000000'}, 'provider_name': None, 'previous_errors': [{'code': 429, 'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day'}, {'code': 429, 'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day'}, {'code': 429, 'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day'}, {'code': 429, 'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day'}, {'code': 429, 'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests p

---
## 4. Streaming — A **Common Function** for Google · NVIDIA · OpenRouter

The LangChain model returned by `utils.get_llm(provider)` supports `.stream()` regardless of provider
(`ChatGoogleGenerativeAI`, `ChatNVIDIA`, and `ChatOpenAI` all have `.stream()`/`.astream()`). So **a single
function** lets us see token streaming from Google, NVIDIA, and OpenRouter alike.

- **Reasoning models** (e.g. NVIDIA `nvidia/nemotron-...`) also stream their thinking process via
  `additional_kwargs['reasoning_content']` (the function below handles both). Not applicable to Gemini.
- Chunk `content` differs between str/list depending on the provider, so we normalize it with `to_text()`. However, **for streaming
  you must pass `strip=False`** — NVIDIA/OpenRouter stream **token by token**, so the default strip would cut the leading space of a token (`" user"`),
  causing words to run together (Gemini is less affected because it sends large chunks).

In [6]:
def stream_answer(provider, messages=PROMPT):
    """Print the response from the given provider as a token stream (shared by Google/NVIDIA/OpenRouter)."""
    llm = utils.get_llm(provider)                 # provider-agnostic LangChain model
    for chunk in llm.stream(messages):
        ak = getattr(chunk, "additional_kwargs", None) or {}
        if ak.get("reasoning_content"):            # thinking process of a reasoning model
            print(ak["reasoning_content"], end="")
        # Streaming prints each chunk (token), so preserve whitespace with strip=False.
        #   NVIDIA/OpenRouter stream token by token, so with the default strip the leading space of a token (" user")
        #   gets cut, making 'wordsruntogether'. Gemini sends large chunks, so it is less affected.
        print(to_text(chunk.content, strip_thinking=False, strip=False), end="")  # answer tokens
    print()

# Stream only from services whose keys are set
targets = []
if GOOGLE_API_KEY:
    targets.append(("google", "Google Gemini"))
if NVIDIA_API_KEY:
    targets.append(("nvidia", f"NVIDIA {utils.NVIDIA_MODEL}"))
if OPENROUTER_API_KEY:
    targets.append(("openrouter", f"OpenRouter {utils.OPENROUTER_MODEL}"))

for provider, label in targets:
    print(f"\n[Streaming · {label}]")
    try:
        stream_answer(provider)
    except Exception as e:
        print(f"  (streaming failed) {type(e).__name__}: {e}")
if not targets:
    print("No keys are set, so the streaming example is skipped.")


[스트리밍 · Google Gemini]


에이전틱 AI는 단순히 명령을 수행하는 것을 넘어, 스스로 목표를 설정하고 계획을 세워 자율적으로 문제를 해결하는 지능형 시스템입니다.

[스트리밍 · NVIDIA meta/llama-3.1-8b-instruct]


에이전틱(Agentic) AI는 사용자의 의도와 목표를 달성하기 위해 행동하는 인공지능을 말합니다.



[스트리밍 · OpenRouter openrouter/free]


  (스트리밍 실패) RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1785024000000'}, 'provider_name': None, 'previous_errors': [{'code': 429, 'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day'}, {'code': 429, 'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day'}, {'code': 429, 'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day'}, {'code': 429, 'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day'}, {'code': 429, 'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day'}, {'code'

---
## 5. Same Question · Comparing Final Responses by Service

View side by side the responses from the three free services to the same prompt (received in §1, §2, §3 above; configured services only).
For OpenRouter, the response from the LangChain path (§3-B) is used.

In [7]:
print('Question:', PROMPT[-1].content)
print('=' * 70)
comparison = [
    ('Google Gemini', gemini_answer),
    (f'NVIDIA {NVIDIA_MODEL}', nvidia_answer),
    (f'OpenRouter {utils.OPENROUTER_MODEL}', openrouter_lc_answer),
]
for name, ans in comparison:
    print(f'\n[{name}]')
    print(' ', ans if ans else '(skipped: key not set / call failed)')

질문: 에이전틱(Agentic) AI를 한 문장으로 설명해줘.

[Google Gemini]
  에이전틱 AI는 단순히 명령을 수행하는 것을 넘어, 스스로 목표를 설정하고 계획을 세워 자율적으로 문제를 해결하는 지능형 시스템입니다.

[NVIDIA meta/llama-3.1-8b-instruct]
  에이전틱(Agentic) AI는 사용자의 의도와 목표를 달성하기 위해 행동하는 인공지능을 말합니다.

[OpenRouter openrouter/free]
  (키 미설정/호출 실패로 건너뜀)


---
## Summary

| Service | `.env` key | Access code |
|---|---|---|
| **Google AI Studio** (optional) | `GOOGLE_API_KEY` | `utils.get_llm('google')` (= `ChatGoogleGenerativeAI`) |
| **NVIDIA build** ⭐Default | `NVIDIA_API_KEY` (+`NVIDIA_MODEL`) | `utils.get_llm('nvidia')` (= `ChatNVIDIA`) |
| **OpenRouter** ⭐Comparison | `OPENROUTER_API_KEY` (+`OPENROUTER_MODEL`) | SDK: `OpenAI(base_url=...)` · LangChain: `utils.get_llm('openrouter')` (= `ChatOpenAI`) |

- All three services can be started for **free**, and via `utils.get_llm()` they are handled the same way as other models (`.invoke()` / `.stream()`).
- Differences in response format (such as Gemini's list content) are absorbed by `agentic_lib.bootstrap.to_text()`.
- Because **OpenRouter** is OpenAI-compatible, both **calling the OpenRouter SDK (OpenAI SDK) directly** and **LangChain (`ChatOpenAI`)** are possible.
  The default model `openrouter/free` (Free Models Router) automatically selects a free model, and the free limit is 50 requests per day (1,000 requests with 10+ credits).
- The course default is **NVIDIA build** (`LLM_PROVIDER=nvidia`), and a single `LLM_PROVIDER` line in `.env`
  switches between `nvidia` ↔ `openrouter` (↔ optional `google`). NVIDIA build is also used in NAT (the `notebooks/NAT_Tutorial/` folder) as `_type: nim`.

### References
- Google AI Studio: https://aistudio.google.com
- NVIDIA build: https://build.nvidia.com  ·  LangChain connector: `langchain-nvidia-ai-endpoints`
- OpenRouter: https://openrouter.ai  ·  Free Models Router: https://openrouter.ai/docs/cookbook/get-started/free-models-router-playground